In [ ]:
!pip install -q "transformers==4.40.2" "huggingface_hub==0.23.4"

In [ ]:
import os, json, time, copy
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import f1_score, accuracy_score

import transformers.utils.hub as _hub
_hub.list_repo_templates = lambda *a, **kw: []

from transformers import BlipProcessor, BlipForImageTextRetrieval

In [ ]:
import random

# ── Dataset paths ──────────────────────────────────────────────────────────────
BASE_INPUT   = "/kaggle/input/datasets/youssefelghandour11/full-dataset"
IMAGES_ROOT  = f"{BASE_INPUT}/images"
WORKING_DIR  = "/kaggle/working"

TRAIN_LABELS = f"{BASE_INPUT}/merged_balanced/train.json"
TRAIN_META   = f"{BASE_INPUT}/metadata/train.json"
VAL_LABELS   = f"{BASE_INPUT}/merged_balanced/val.json"
VAL_META     = f"{BASE_INPUT}/metadata/val.json"

# ── Checkpoint to resume from (best so far = epoch 7) ─────────────────────────
RESUME_CKPT  = "/kaggle/input/datasets/youssefelghandour11/blip-checkpoint/blip_itm_finetuned_best.pt"
BEST_CKPT    = f"{WORKING_DIR}/blip_itm_finetuned_best.pt"
HISTORY_JSON = f"{WORKING_DIR}/training_history.json"

MODEL_NAME   = "Salesforce/blip-itm-base-coco"

# ── Resume state ───────────────────────────────────────────────────────────────
START_EPOCH       = 9    # next epoch to run
BEST_VAL_F1       = 0.0  # reset to 0 — now tracking val F1, not train F1
EPOCHS_NO_IMPROVE = 0    # fresh patience counter for val-based checkpointing

# ── Hyperparameters (must match original run) ─────────────────────────────────
BATCH_SIZE        = 32
NUM_WORKERS       = 4
MAX_EPOCHS        = 30
PATIENCE          = 5
LR_HEAD           = 1e-5
LR_ENCODER        = 5e-6
UNFREEZE_LAYERS   = 2
HARD_NEG_COPIES   = 2
HARD_NEG_CAP_FRAC = 0.30   # cap hard samples at 30% of base train set
LOW_CONF_LOW      = 0.4
LOW_CONF_HIGH     = 0.6

In [ ]:
# Seed the history file with the completed epochs so it stays consistent
PRIOR_HISTORY = [
  {"epoch": 1, "train_loss": 0.723175, "train_acc": 0.535555, "train_f1": 0.535732, "hard_samples": 0,     "next_train_size": 71072,  "elapsed_sec": 1583.1},
  {"epoch": 2, "train_loss": 0.68533,  "train_acc": 0.547149, "train_f1": 0.543715, "hard_samples": 65073, "next_train_size": 201218, "elapsed_sec": 1584.2},
  {"epoch": 3, "train_loss": 0.690705, "train_acc": 0.533014, "train_f1": 0.565097, "hard_samples": 68640, "next_train_size": 208352, "elapsed_sec": 4487.1},
  {"epoch": 4, "train_loss": 0.668732, "train_acc": 0.584165, "train_f1": 0.579846, "hard_samples": 50106, "next_train_size": 171284, "elapsed_sec": 4653.7},
  {"epoch": 5, "train_loss": 0.680737, "train_acc": 0.551207, "train_f1": 0.532122, "hard_samples": 54628, "next_train_size": 180328, "elapsed_sec": 3814.8},
  {"epoch": 6, "train_loss": 0.636527, "train_acc": 0.619937, "train_f1": 0.647775, "hard_samples": 36679, "next_train_size": 144430, "elapsed_sec": 4017.5},
  {"epoch": 7, "train_loss": 0.63425,  "train_acc": 0.603282, "train_f1": 0.652929, "hard_samples": 39241, "next_train_size": 149554, "elapsed_sec": 3218.5},
  {"epoch": 8, "train_loss": 0.578952, "train_acc": 0.66354,  "train_f1": 0.53363,  "hard_samples": 31144, "next_train_size": 133360, "elapsed_sec": 3331.7},
]

with open(HISTORY_JSON, "w") as f:
    json.dump(PRIOR_HISTORY, f, indent=2)
print(f"History seeded with {len(PRIOR_HISTORY)} epochs.")

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

In [ ]:
def build_image_path(raw_path):
    stripped = raw_path.removeprefix("visual_news/")
    return os.path.join(IMAGES_ROOT, stripped)


def load_split(labels_path, meta_path, split_name):
    with open(labels_path) as f:
        annotations = json.load(f)["annotations"]
    with open(meta_path) as f:
        metadata = json.load(f)

    samples, missing_meta, missing_file = [], 0, 0
    for ann in annotations:
        key = str(ann["image_id"])
        if key not in metadata:
            missing_meta += 1
            continue
        meta     = metadata[key]
        img_path = build_image_path(meta["image_path"])
        if not os.path.exists(img_path):
            missing_file += 1
            continue
        samples.append({
            "image_id":   ann["image_id"],
            "image_path": img_path,
            "caption":    meta.get("caption", ""),
            "label":      int(ann["falsified"]),
        })

    print(f"[{split_name}] loaded={len(samples)} | "
          f"missing_meta={missing_meta} | missing_file={missing_file}")
    return samples

In [ ]:
class ITMDataset(Dataset):
    def __init__(self, samples, processor, max_text_len=128):
        self.samples       = samples
        self.processor     = processor
        self.max_text_len  = max_text_len
        self.corrupt_count = 0

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        try:
            image = Image.open(s["image_path"]).convert("RGB")
        except (UnidentifiedImageError, OSError):
            self.corrupt_count += 1
            image = Image.new("RGB", (384, 384))

        encoding = self.processor(
            images=image,
            text=s["caption"],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_text_len,
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"]     = torch.tensor(s["label"], dtype=torch.long)
        item["sample_idx"] = torch.tensor(idx, dtype=torch.long)
        return item


def make_loader(samples, processor, shuffle=True, batch_size=BATCH_SIZE):
    ds = ITMDataset(samples, processor)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    ), ds

In [ ]:
def freeze_model(model):
    for p in model.parameters():
        p.requires_grad = False


def unfreeze_last_n_layers(encoder, n):
    for layer in list(encoder.encoder.layer)[-n:]:
        for p in layer.parameters():
            p.requires_grad = True


def build_model_and_optimizer():
    model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME)

    freeze_model(model)
    unfreeze_last_n_layers(model.vision_model, UNFREEZE_LAYERS)
    unfreeze_last_n_layers(model.text_encoder, UNFREEZE_LAYERS)
    for name, p in model.named_parameters():
        if any(k in name for k in ["itm_head", "vision_proj", "text_proj"]):
            p.requires_grad = True

    encoder_params = [p for n, p in model.named_parameters()
                      if p.requires_grad and
                      not any(k in n for k in ["itm_head", "vision_proj", "text_proj"])]
    head_params    = [p for n, p in model.named_parameters()
                      if p.requires_grad and
                      any(k in n for k in ["itm_head", "vision_proj", "text_proj"])]

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

    optimizer = torch.optim.AdamW([
        {"params": encoder_params, "lr": LR_ENCODER},
        {"params": head_params,    "lr": LR_HEAD},
    ], weight_decay=1e-4)

    return model, optimizer

In [ ]:
@torch.no_grad()
def find_hard_samples(model, samples, processor, device):
    model.eval()
    ds     = ITMDataset(samples, processor)
    loader = DataLoader(ds, batch_size=64, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

    hard_indices = []
    offset = 0

    for batch in loader:
        pixel_values   = batch["pixel_values"].to(device)
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        with autocast():
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_itm_head=True,
            )

        probs     = torch.softmax(outputs.itm_score.float(), dim=-1)
        preds     = probs.argmax(dim=-1)
        fake_prob = probs[:, 1]
        is_hard   = ((preds != labels) | ((fake_prob >= LOW_CONF_LOW) & (fake_prob <= LOW_CONF_HIGH))).cpu().numpy()

        for i, hard in enumerate(is_hard):
            if hard:
                hard_indices.append(offset + i)
        offset += len(labels)

    # Cap at 30% of base train set to prevent runaway oversampling
    cap = int(HARD_NEG_CAP_FRAC * len(samples))
    if len(hard_indices) > cap:
        hard_indices = random.sample(hard_indices, cap)

    return hard_indices


def build_oversampled_samples(base_samples, hard_indices, copies=HARD_NEG_COPIES):
    hard_samples = [base_samples[i] for i in hard_indices]
    return base_samples + hard_samples * copies

In [ ]:
def run_epoch(model, loader, optimizer, scaler, device, is_train):
    model.train() if is_train else model.eval()
    criterion = nn.CrossEntropyLoss()

    total_loss, all_preds, all_labels = 0.0, [], []
    first_batch = True

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in loader:
            pixel_values   = batch["pixel_values"].to(device)
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            with autocast():
                outputs = model(
                    pixel_values=pixel_values,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    use_itm_head=True,
                )
                loss = criterion(outputs.itm_score, labels)

            if is_train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * labels.size(0)
            preds = outputs.itm_score.argmax(dim=-1).detach().cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

            if first_batch and is_train:
                first_batch = False
                for i in range(torch.cuda.device_count()):
                    mem = torch.cuda.memory_allocated(i) / 1e9
                    res = torch.cuda.memory_reserved(i) / 1e9
                    print(f"  GPU {i}: allocated={mem:.2f}GB  reserved={res:.2f}GB")

    n        = len(all_labels)
    avg_loss = total_loss / n
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average="binary", zero_division=0)
    return avg_loss, acc, f1

In [ ]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}  |  GPUs: {torch.cuda.device_count()}")

    train_samples = load_split(TRAIN_LABELS, TRAIN_META, "TRAIN")
    val_samples   = load_split(VAL_LABELS,   VAL_META,   "VAL")

    processor = BlipProcessor.from_pretrained(MODEL_NAME)

    model, optimizer = build_model_and_optimizer()

    print(f"Loading checkpoint: {RESUME_CKPT}")
    state_dict = torch.load(RESUME_CKPT, map_location="cpu")
    model.load_state_dict(state_dict)
    print("Checkpoint loaded.")

    if torch.cuda.device_count() > 1:
        print(f"Wrapping model in DataParallel across {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    model.to(device)

    scaler = GradScaler()

    with open(HISTORY_JSON) as f:
        history = json.load(f)
    best_val_f1       = BEST_VAL_F1
    epochs_no_improve = EPOCHS_NO_IMPROVE

    # Recompute hard negatives from the loaded checkpoint before epoch START_EPOCH
    print("\nRecomputing hard negatives for resumed epoch...")
    hard_indices          = find_hard_samples(model, train_samples, processor, device)
    current_train_samples = (
        build_oversampled_samples(train_samples, hard_indices)
        if hard_indices else train_samples
    )
    print(f"hard_samples={len(hard_indices)} | "
          f"starting_train_size={len(current_train_samples)}")

    print("\n" + "\u2550"*65)
    print(f"Resuming training from epoch {START_EPOCH}")
    print(f"Best val F1 so far: {best_val_f1:.4f} | "
          f"Epochs without improvement: {epochs_no_improve}/{PATIENCE}")
    print("\u2550"*65)

    for epoch in range(START_EPOCH, MAX_EPOCHS + 1):
        t0 = time.time()
        print(f"\n\u2500\u2500 Epoch {epoch}/{MAX_EPOCHS} "
              f"| train_set_size={len(current_train_samples)} \u2500\u2500")

        train_loader, train_ds = make_loader(
            current_train_samples, processor, shuffle=True)
        train_loss, train_acc, train_f1 = run_epoch(
            model, train_loader, optimizer, scaler, device, is_train=True)

        if train_ds.corrupt_count:
            print(f"  Corrupt images skipped: {train_ds.corrupt_count}")

        # Val evaluation
        val_loader, _ = make_loader(val_samples, processor, shuffle=False)
        val_loss, val_acc, val_f1 = run_epoch(
            model, val_loader, None, None, device, is_train=False)

        elapsed = time.time() - t0
        print(f"  train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  train_F1={train_f1:.4f}")
        print(f"  val_loss={val_loss:.4f}    val_acc={val_acc:.4f}    val_F1={val_f1:.4f}  ({elapsed:.0f}s)")

        # Hard negative mining (capped at 30%)
        hard_indices = find_hard_samples(model, train_samples, processor, device)
        num_hard     = len(hard_indices)
        current_train_samples = (
            build_oversampled_samples(train_samples, hard_indices)
            if hard_indices else train_samples
        )
        print(f"  hard_samples={num_hard} | "
              f"next_epoch_train_size={len(current_train_samples)}")

        # Checkpoint on val F1
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            inner = model.module if hasattr(model, "module") else model
            torch.save(copy.deepcopy(inner.state_dict()), BEST_CKPT)
            print(f"  \u2713 New best val F1={best_val_f1:.4f} \u2014 checkpoint saved")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            print(f"  No improvement ({epochs_no_improve}/{PATIENCE})")

        history.append({
            "epoch":           epoch,
            "train_loss":      round(train_loss, 6),
            "train_acc":       round(train_acc, 6),
            "train_f1":        round(train_f1, 6),
            "val_loss":        round(val_loss, 6),
            "val_acc":         round(val_acc, 6),
            "val_f1":          round(val_f1, 6),
            "hard_samples":    num_hard,
            "next_train_size": len(current_train_samples),
            "elapsed_sec":     round(elapsed, 1),
        })
        with open(HISTORY_JSON, "w") as f:
            json.dump(history, f, indent=2)

        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping triggered after epoch {epoch}.")
            break

    print("\n" + "\u2550"*95)
    print(f"{'Epoch':>6} {'TrLoss':>8} {'TrAcc':>7} {'TrF1':>7} "
          f"{'VaLoss':>8} {'VaAcc':>7} {'VaF1':>7} {'Hard':>7} {'NextSz':>8} {'Time':>7}")
    print("\u2500"*95)
    for r in history:
        vl = r.get('val_loss', float('nan'))
        va = r.get('val_acc',  float('nan'))
        vf = r.get('val_f1',   float('nan'))
        print(f"{r['epoch']:>6} {r['train_loss']:>8.4f} {r['train_acc']:>7.4f} {r['train_f1']:>7.4f} "
              f"{vl:>8.4f} {va:>7.4f} {vf:>7.4f} "
              f"{r['hard_samples']:>7} {r['next_train_size']:>8} {r['elapsed_sec']:>6.0f}s")
    print("\u2550"*95)
    print(f"Best val F1: {best_val_f1:.4f}")
    print(f"Best checkpoint: {BEST_CKPT}")


main()